In [1]:
import re
import ast
import requests
from collections import defaultdict

# ── CONFIG ─────────────────────────────────────────────────────────────────────

URLS_PY_RAW = "https://raw.githubusercontent.com/PokeAPI/pokeapi/master/pokemon_v2/urls.py"
API_PY_RAW  = "https://raw.githubusercontent.com/PokeAPI/pokeapi/master/pokemon_v2/api.py"
API_BASE    = "/api/v2/"

# ── UTILITIES ──────────────────────────────────────────────────────────────────

def fetch_raw(url: str) -> str:
    r = requests.get(url)
    r.raise_for_status()
    return r.text

# ── STEP 1: PARSE urls.py ───────────────────────────────────────────────────────

def parse_urls_py(source: str):
    """
    Returns:
      - resources: list of (prefix, ViewSetClassName)
      - manual:    list of (raw_regex, ViewClassName)
    """
    resources = []
    manual    = []
    # router.register(r"prefix", ResourceClass)
    for m in re.finditer(
        r'router\.register\(\s*r"(.*?)"\s*,\s*([A-Za-z0-9_]+)\s*\)',
        source,
    ):
        prefix, cls = m.groups()
        resources.append((prefix, cls))

    # url(r"^api/v2/some/regex", SomeView.as_view())
    for m in re.finditer(
        r'url\(\s*r"(.*?)"\s*,\s*([A-Za-z0-9_]+)\.as_view\(\)',
        source,
    ):
        raw_regex, cls = m.groups()
        manual.append((raw_regex, cls))

    return resources, manual

# ── STEP 2: PARSE api.py ────────────────────────────────────────────────────────

def parse_api_py_methods(source: str):
    """
    Returns a dict { ViewClassName: [ "GET", "POST", ... ] }.
    Falls back to GET if no methods found.
    """
    tree = ast.parse(source)
    methods = defaultdict(list)

    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            cls_name = node.name
            for fn in node.body:
                if isinstance(fn, ast.FunctionDef):
                    name = fn.name.lower()
                    if name in ("get","post","put","patch","delete"):
                        methods[cls_name].append(name.upper())

    # Default: if it's a DRF ViewSet but no overrides, we'll infer below
    return methods

# ── STEP 3: INFER HTTP methods for ViewSets ────────────────────────────────────

def infer_viewset_methods(source: str, cls_names):
    """
    Scans the imports and base classes in source to figure out
    which of our resource classes inherit from ReadOnlyModelViewSet
    vs ModelViewSet vs APIView.
    """
    tree = ast.parse(source)
    inheritance = {}
    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name in cls_names:
            bases = [getattr(b, 'attr', getattr(b, 'id', None)) for b in node.bases]
            # common conventions in api.py:
            if "ReadOnlyModelViewSet" in bases:
                inheritance[node.name] = "readonly"
            elif "ModelViewSet" in bases:
                inheritance[node.name] = "full"
            elif "APIView" in bases:
                inheritance[node.name] = "api_view"
            else:
                inheritance[node.name] = "unknown"
    return inheritance

# ── STEP 4: ASSEMBLE ALL ROUTES ────────────────────────────────────────────────

def build_route_table(resources, manual, api_methods, inheritance_map):
    """
    Yields tuples: (path, view_class, [methods...])
    """
    out = []
    # 1) resources via DefaultRouter:
    for prefix, cls in resources:
        kind = inheritance_map.get(cls, "")
        if kind == "readonly":
            methods = ["GET"]
        elif kind == "full":
            methods = ["GET","POST","PUT","PATCH","DELETE"]
        else:
            # fallback to what was explicitly overridden
            methods = api_methods.get(cls, ["GET"])
        # base & detail
        out.append((f"{API_BASE}{prefix}/",          cls, methods))
        out.append((f"{API_BASE}{prefix}/{{pk}}/",   cls, methods))
    # 2) manual url(...) entries
    for raw_regex, cls in manual:
        # strip leading ^ and trailing $
        regex = raw_regex.lstrip("^").rstrip("$")
        path  = regex.replace("\\d+", "{n}")  # simple var
        out.append((path, cls, api_methods.get(cls, ["GET"])))
    return out

# ── MAIN ───────────────────────────────────────────────────────────────────────

urls_py = fetch_raw(URLS_PY_RAW)
api_py  = fetch_raw(API_PY_RAW)

resources, manual = parse_urls_py(urls_py)
api_methods       = parse_api_py_methods(api_py)
inheritance_map   = infer_viewset_methods(api_py, [c for _,c in resources] + [cls for _,cls in manual])

table = build_route_table(resources, manual, api_methods, inheritance_map)

# Print
print(f"{'PATH':40} │ {'VIEW':30} │ METHODS")
print("-"*90)
for path, view, methods in sorted(table):
    print(f"{path:40} │ {view:30} │ {', '.join(methods)}")


PATH                                     │ VIEW                           │ METHODS
------------------------------------------------------------------------------------------
/api/v2/ability/                         │ AbilityResource                │ GET
/api/v2/ability/{pk}/                    │ AbilityResource                │ GET
/api/v2/berry-firmness/                  │ BerryFirmnessResource          │ GET
/api/v2/berry-firmness/{pk}/             │ BerryFirmnessResource          │ GET
/api/v2/berry-flavor/                    │ BerryFlavorResource            │ GET
/api/v2/berry-flavor/{pk}/               │ BerryFlavorResource            │ GET
/api/v2/berry/                           │ BerryResource                  │ GET
/api/v2/berry/{pk}/                      │ BerryResource                  │ GET
/api/v2/characteristic/                  │ CharacteristicResource         │ GET
/api/v2/characteristic/{pk}/             │ CharacteristicResource         │ GET
/api/v2/contest-effect/  

In [20]:
import re
import ast
import asyncio
import aiohttp
from collections import defaultdict

import polars as pl
import duckdb
import pyarrow as pa
import pyarrow as pa

# ── CONFIG ─────────────────────────────────────────────────────────────────────
URLS_PY_RAW = "https://raw.githubusercontent.com/PokeAPI/pokeapi/master/pokemon_v2/urls.py"
API_PY_RAW  = "https://raw.githubusercontent.com/PokeAPI/pokeapi/master/pokemon_v2/api.py"
API_BASE    = "/api/v2/"
POKEAPI_BASE = "https://pokeapi.co"

# ── SYNC UTILS ─────────────────────────────────────────────────────────────────
import requests
def fetch_raw(url: str) -> str:
    r = requests.get(url)
    r.raise_for_status()
    return r.text

# ── PARSING ─────────────────────────────────────────────────────────────────────
def parse_urls_py(source: str):
    resources, manual = [], []
    for m in re.finditer(r'router\.register\(\s*r"(.*?)"\s*,\s*([A-Za-z0-9_]+)\s*\)', source):
        resources.append(m.groups())
    for m in re.finditer(r'url\(\s*r"(.*?)"\s*,\s*([A-Za-z0-9_]+)\.as_view\(\)', source):
        manual.append(m.groups())
    return resources, manual

def parse_api_py_methods(source: str):
    tree = ast.parse(source)
    methods = defaultdict(list)
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            cls_name = node.name
            for fn in node.body:
                if isinstance(fn, ast.FunctionDef) and fn.name.lower() in {"get", "post", "put", "patch", "delete"}:
                    methods[cls_name].append(fn.name.upper())
    return methods

def infer_viewset_methods(source: str, cls_names):
    tree = ast.parse(source)
    inheritance = {}
    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name in cls_names:
            bases = [getattr(b, 'attr', getattr(b, 'id', None)) for b in node.bases]
            if "ReadOnlyModelViewSet" in bases:
                inheritance[node.name] = "readonly"
            elif "ModelViewSet" in bases:
                inheritance[node.name] = "full"
            elif "APIView" in bases:
                inheritance[node.name] = "api_view"
            else:
                inheritance[node.name] = "unknown"
    return inheritance

def build_route_table(resources, manual, api_methods, inheritance_map):
    table = []
    for prefix, cls in resources:
        kind = inheritance_map.get(cls, "")
        methods = {
            "readonly": ["GET"],
            "full": ["GET", "POST", "PUT", "PATCH", "DELETE"]
        }.get(kind, api_methods.get(cls, ["GET"]))
        table.append((f"{API_BASE}{prefix}/", cls, methods))
        table.append((f"{API_BASE}{prefix}/{{pk}}/", cls, methods))
    for raw_regex, cls in manual:
        path = raw_regex.lstrip("^").rstrip("$").replace("\\d+", "{n}")
        table.append((path, cls, api_methods.get(cls, ["GET"])))
    return table

# ── ASYNC FETCH ────────────────────────────────────────────────────────────────
async def fetch_paginated_data(session, route):
    url = f"{POKEAPI_BASE}{route}"
    results = []
    try:
        while url:
            async with session.get(url) as resp:
                if resp.status != 200:
                    break
                data = await resp.json()
                results.extend(data.get("results", []))
                url = data.get("next", None)
    except Exception as e:
        print(f"⚠️ Failed route: {route} ({e})")
    return (route, results)

def persist_results_to_duckdb(results: dict, db_path: str = "pokeapi_data.duckdb"):
    conn = duckdb.connect(db_path)

    for route, items in results.items():
        if not items:
            print(f"⚠️  Skipping {route}: empty result set")
            continue

        try:
            dict_items = [item for item in items if isinstance(item, dict)]

            key_sets = [frozenset(d.keys()) for d in dict_items]
            unique_key_sets = set(key_sets)

            if len(unique_key_sets) > 1:
                print(f"⚠️  Inconsistent keys in {route}: found {len(unique_key_sets)} different key sets")

            all_keys = sorted(set().union(*key_sets))
            normalized_dicts = [{k: d.get(k, None) for k in all_keys} for d in dict_items]
            table = pa.Table.from_pylist(normalized_dicts)

            table_name = route.rstrip("/").split("/")[-1].replace("-", "_")
            view_name = f"_tmp_{table_name}"

            conn.register(view_name, table)
            conn.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM {view_name}")
            conn.unregister(view_name)

            print(f"✅ Wrote {table_name}")

        except Exception as e:
            print(f"❌ Failed to write {route}: {e}")
    
    conn.commit()
    conn.close()

# ── MAIN ───────────────────────────────────────────────────────────────────────
from tqdm.asyncio import tqdm_asyncio

async def main(persist: bool = True):
    urls_py = fetch_raw(URLS_PY_RAW)
    api_py  = fetch_raw(API_PY_RAW)

    resources, manual = parse_urls_py(urls_py)
    api_methods = parse_api_py_methods(api_py)
    cls_names = [c for _, c in resources] + [cls for _, cls in manual]
    inheritance_map = infer_viewset_methods(api_py, cls_names)

    route_table = build_route_table(resources, manual, api_methods, inheritance_map)

    get_routes = sorted(set([
        path for path, _, methods in route_table
        if path.endswith("/") and "GET" in methods and "{" not in path
    ]))

    async with aiohttp.ClientSession() as session:
        tasks = [fetch_paginated_data(session, route) for route in get_routes]
        results = await tqdm_asyncio.gather(*tasks, desc="Fetching async")

    flat_results = dict(results)

    if persist:
        persist_results_to_duckdb(flat_results)

    return flat_results

results = await main(persist=True)



Fetching async: 100%|██████████| 48/48 [00:05<00:00,  9.49it/s]

✅ Wrote ability
✅ Wrote berry_firmness
✅ Wrote berry_flavor
✅ Wrote berry
✅ Wrote characteristic
✅ Wrote contest_effect
✅ Wrote contest_type
✅ Wrote egg_group
✅ Wrote encounter_condition_value
✅ Wrote encounter_condition
✅ Wrote encounter_method
✅ Wrote evolution_chain
✅ Wrote evolution_trigger
✅ Wrote gender
✅ Wrote generation
✅ Wrote growth_rate
✅ Wrote item_attribute
✅ Wrote item_category
✅ Wrote item_fling_effect
✅ Wrote item_pocket
✅ Wrote item
✅ Wrote language
✅ Wrote location_area
✅ Wrote location
✅ Wrote machine
✅ Wrote move_ailment
✅ Wrote move_battle_style
✅ Wrote move_category
✅ Wrote move_damage_class
✅ Wrote move_learn_method
✅ Wrote move_target
✅ Wrote move
✅ Wrote nature
✅ Wrote pal_park_area
✅ Wrote pokeathlon_stat
✅ Wrote pokedex
✅ Wrote pokemon_color
✅ Wrote pokemon_form
✅ Wrote pokemon_habitat
✅ Wrote pokemon_shape
✅ Wrote pokemon_species
✅ Wrote pokemon
✅ Wrote region
✅ Wrote stat
✅ Wrote super_contest_effect
✅ Wrote type
✅ Wrote version_group
✅ Wrote version


In [ ]:
import asyncio
import aiohttp
import logging
import re
from urllib.parse import urlparse
from pathlib import Path
from bs4 import BeautifulSoup
import pyarrow as pa
import duckdb

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("digimon-async")

# Globals
DUCKDB_FILE = "digimon_data.duckdb"
MAX_CONCURRENT_REQUESTS = 10

# DuckDB connection (global for now)
conn = duckdb.connect(DUCKDB_FILE)
created_tables = set()

# ----------------------- UTILITIES -----------------------
def sanitize_table_name(name: str) -> str:
    return re.sub(r'\W+', '_', name).lower()

def flatten_dict(d, parent_key='', sep='_'):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

# ---------------------- DUCKDB I/O -----------------------
def create_table(data, table_name):
    if not data or table_name in created_tables:
        return

    all_columns = sorted({k for item in data for k in flatten_dict(item).keys()})
    columns_str = ", ".join([f'"{col}" TEXT' for col in all_columns])
    query = f'CREATE TABLE IF NOT EXISTS "{table_name}" ({columns_str});'
    conn.execute(query)
    created_tables.add(table_name)

def insert_arrow(data, table_name):
    if not data:
        return
    flat = [flatten_dict(d) for d in data if isinstance(d, dict)]
    if not flat:
        return

    batch = pa.Table.from_pylist(flat)
    conn.register("temp_table", batch)
    try:
        conn.execute(f'INSERT INTO "{table_name}" SELECT * FROM temp_table;')
    except Exception as e:
        logger.error(f"Insert failed for {table_name}: {e}")
    finally:
        conn.unregister("temp_table")

# --------------------- ENDPOINT SCRAPER ---------------------
def extract_urls_from_docs(docs_url: str, use_cache=True):
    html = requests.get(docs_url).text
    soup = BeautifulSoup(html, "html.parser")
    endpoints = set()

    for h4 in soup.find_all("h4", class_="svelte-c41qbh"):
        if h4.text.startswith("Endpoint:"):
            line = h4.text.replace("Endpoint:", "").strip()
            if "{" in line and "or" in line:
                match = re.match(r"(.*)\{.*?\}.*?or\s*\{(.*?)\}", line)
                if match:
                    prefix, alt = match.groups()
                    endpoints.add(f"{prefix}{{id}}")
                    endpoints.add(f"{prefix}{{{alt}}}")
                else:
                    endpoints.add(line)
            else:
                endpoints.add(line)

    final_list = sorted(url for url in endpoints if "{" not in url)
    return final_list

# -------------------- ASYNC FETCHER --------------------
async def fetch_data(session, url, sem):
    async with sem:
        try:
            async with session.get(url, timeout=10) as resp:
                if resp.status != 200:
                    logger.warning(f"{url} returned {resp.status}")
                    return url, []
                json_data = await resp.json()
                content = json_data.get("content")
                if isinstance(content, dict):
                    return url, [content]
                return url, content or []
        except Exception as e:
            logger.error(f"Error fetching {url}: {e}")
            return url, []

# -------------------- MAIN RUNNER --------------------
async def main():
    docs_url = "https://digi-api.com/#docs"
    base_urls = extract_urls_from_docs(docs_url)
    logger.info(f"Found {len(base_urls)} endpoints")

    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async with aiohttp.ClientSession() as session:
        tasks = [fetch_data(session, url if url.startswith("http") else f"http://{url}", sem) for url in base_urls]
        
        results = []

        for coro in asyncio.as_completed(tasks):
            url, data = await coro
            results.append((url, data))
            
            if data:
                table_name = sanitize_table_name(urlparse(url).path.strip("/").split("/")[-1])
                create_table(data, table_name)
                insert_arrow(data, table_name)

    conn.commit()
    conn.close()
    logger.info("Done.")

    return dict(results)

await main()


INFO:digimon-async:Found 6 endpoints
INFO:digimon-async:Done.


{'http://digi-api.com/api/v1/type': [{'name': 'Type',
   'description': "A Digimon's Type indicates what sort of category a Digimon's specific species belongs to. Many of these simply indicate what a Digimon is based on and only come into play under certain situations - some Digimon may have a certain advantage or disadvantage to a Digimon of another type. Or an item will work on a Digimon of one type or not the other.",
   'fields': [{'id': 1,
     'name': 'Reptile',
     'href': 'https://digi-api.com/api/v1/type/1'},
    {'id': 2,
     'name': 'Mythical Beast',
     'href': 'https://digi-api.com/api/v1/type/2'},
    {'id': 3, 'name': 'Angel', 'href': 'https://digi-api.com/api/v1/type/3'},
    {'id': 4,
     'name': 'Amphibian',
     'href': 'https://digi-api.com/api/v1/type/4'},
    {'id': 5,
     'name': 'Giant Bird',
     'href': 'https://digi-api.com/api/v1/type/5'}]}],
 'http://digi-api.com/api/v1/level': [{'name': 'Level',
   'description': 'An Evolution Stage, also referred to 

In [58]:
conn.commit()
conn.close()